# Laboratory 1 - From Pixels to Semantics

**Author:** Edoardo Canti<br>
<br>
**Notebook goal:** This notebook represent Exercise 1.2, using a pretrained network as feature extractor in order to train a classical model for classifcation. This classical model will be used as a lower bound performances baseline:

### Notes
- **Personal thoughts and comments**: during the drafting process of the notebook, will be provided markdowns regarding personal thoughts, ideas or general comments about the projetc design. I think this will be helpful both for me and for who reads (as this will somehow work as "documentation")

- **Notation**: if during the drafting process some details are considered more important than others, in order to evaluate results or to make additional consideration, those will be marked with [TBC.num] (To Be Considered), where num indicates the "identifier" for the final considerations. However I'll try to answer or provide further explainations of all personal thoughts and comments.

- **Use of LLMs**: If AI tools are used for programming, the following information will be provided: which model was used, what question was asked, and whether the model’s response was modified (if not specified, this means the model’s original response was used).

- **Analysis**: personal thoughts and comments are intended as **Analysis**, but since those will be sparsed in the notebook I will summarize them together in a dedicated section at the end of the notebook.

---
# Exercise 1.2: A Stable and Reproducible Baseline

>In this exercise you should implement code to use a pretrained network as a *feature extractor* that, instead of *classifying* images in input, should return the *feature representation* from the last layer of the pretrained model before the classifier. These features, extracted from the train set, should be used to train a *classical* model for classification (e.g. an SVM, a Nearest Neighbor, or a Linear Discriminant classifier from Scikit-learn). Evaluate the performance of this baseline model on the features extracted from the test set.v

In [4]:
import torch
from torchvision.datasets import GTSRB
import yaml # made a config.yaml file for storing config variables
from torchvision.models import list_models, get_model # pretrained models from torch for vision tasks
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as T
import copy
import os

# Importing config file in order to retrieve config variables of interest
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

In [ ]:
BATCH_SIZE = config['STABLE_REPRODUCIBLE_BASELINE']['batch_size']
print(BATCH_SIZE)

512


['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fasterrcnn_mobilenet_v3_large_320_fpn',
 'fasterrcnn_mobilenet_v3_large_fpn',
 'fasterrcnn_resnet50_fpn',
 'fasterrcnn_resnet50_fpn_v2',
 'fcn_resnet101',
 'fcn_resnet50',
 'fcos_resnet50_fpn',
 'googlenet',
 'inception_v3',
 'keypointrcnn_resnet50_fpn',
 'lraspp_mobilenet_v3_large',
 'maskrcnn_resnet50_fpn',
 'maskrcnn_resnet50_fpn_v2',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'quantized_googlenet',
 '

## 0) Importing Data and DataLoader while taking a look at the Model

In [6]:
transform = T.Compose([T.Resize(70), T.RandomCrop((64,64)), T.ToTensor(), T.Normalize(mean = [0.485, 0.456, 0.406], std=[0.229,0.224, 0.225])])
ds_train = GTSRB('_data/', split='train', transform=transform, download=True)
ds_test = GTSRB('_data/', split='test', transform=transform, download=True)
train_loader = DataLoader(ds_train, BATCH_SIZE, shuffle=True)
test_loader = DataLoader(ds_test, BATCH_SIZE, shuffle=True)
model = get_model('resnet18', weights = 'DEFAULT')
model

/Users/edoardocanti/opt/anaconda3/envs/DLA2026/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


100%|██████████| 187M/187M [00:03<00:00, 50.5MB/s] 


Extracting _data/gtsrb/GTSRB-Training_fixed.zip to _data/gtsrb


100%|██████████| 89.0M/89.0M [00:01<00:00, 50.7MB/s]


Extracting _data/gtsrb/GTSRB_Final_Test_Images.zip to _data/gtsrb


100%|██████████| 99.6k/99.6k [00:00<00:00, 735kB/s]


Extracting _data/gtsrb/GTSRB_Final_Test_GT.zip to _data/gtsrb


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 1) Removing the classification head

Since we want to use this model as a **baseline**, we are going to use it as a **Features Extractor**, this means we are not interested in <br>
(fc): Linear(in_features=512, out_features=1000, bias=True)

In [15]:
# We cannot "remove" the classification head, so a good practice is to transorm it in order to apply a function
# that actually doesn't change the previous avgpool layer (which return a 512 representation of the input)
model.fc = torch.nn.Identity() # "overriding the classification head"
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## 2) Getting training and test features

In [16]:
train_feats = []
train_classes = []
model.eval()
for(ims, cls) in train_loader:
    with torch.no_grad():
        train_feats.append(model(ims))
    train_classes.append(cls.T)
train_feats = torch.vstack(train_feats).cpu()
train_classes = torch.concat(train_classes)

In [17]:
print(train_feats.shape)
train_feats

torch.Size([26640, 512])


tensor([[0.0000e+00, 1.0893e+00, 0.0000e+00,  ..., 4.6204e+00, 7.5005e-01,
         2.1087e-01],
        [1.6276e+00, 1.9568e+00, 2.1660e+00,  ..., 1.2524e-01, 2.0587e+00,
         9.6156e-02],
        [1.8178e+00, 3.6854e+00, 1.1247e+01,  ..., 6.9415e-02, 0.0000e+00,
         1.2104e+00],
        ...,
        [1.0188e+00, 0.0000e+00, 1.2055e+00,  ..., 5.6080e-01, 4.7426e-03,
         1.8090e-02],
        [1.7454e+00, 3.7554e-01, 2.5134e+00,  ..., 3.1340e+00, 3.8266e-03,
         1.9075e-01],
        [5.6120e-02, 0.0000e+00, 5.6783e-02,  ..., 7.6366e-03, 7.5517e-01,
         0.0000e+00]])

In [18]:
test_feats = []
test_classes = []
model.eval()
for(ims, cls) in test_loader:
    with torch.no_grad():
        test_feats.append(model(ims))
    test_classes.append(cls.T)
test_feats = torch.vstack(test_feats).cpu()
test_classes = torch.concat(test_classes)

In [19]:
print(test_feats.shape)
test_feats

torch.Size([12630, 512])


tensor([[2.4844, 0.0000, 1.3310,  ..., 2.3725, 1.1412, 0.0000],
        [0.0000, 0.0000, 1.9229,  ..., 0.0078, 1.5556, 0.0000],
        [0.9175, 0.0000, 0.0000,  ..., 4.7251, 0.8252, 0.9446],
        ...,
        [0.2668, 0.9021, 0.0000,  ..., 2.8666, 1.5924, 0.5012],
        [2.9587, 4.5710, 3.6779,  ..., 1.3640, 1.4455, 0.0000],
        [0.4072, 6.3276, 2.6729,  ..., 2.3226, 0.0000, 0.0000]])

## 3) The baseline itself

In [20]:
from sklearn.svm import SVC

svc = SVC(kernel='linear')
svc.fit(train_feats, train_classes)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [21]:
from sklearn.metrics import classification_report
#print(classification_report(svc.predict(test_feats), test_classes))
print(classification_report(test_classes, svc.predict(test_feats)))

              precision    recall  f1-score   support

           0       0.19      0.23      0.21        60
           1       0.48      0.66      0.56       720
           2       0.42      0.43      0.43       750
           3       0.33      0.36      0.35       450
           4       0.57      0.54      0.56       660
           5       0.49      0.40      0.44       630
           6       0.95      0.83      0.89       150
           7       0.61      0.55      0.58       450
           8       0.49      0.40      0.44       450
           9       0.89      0.77      0.83       480
          10       0.84      0.87      0.86       660
          11       0.45      0.57      0.50       420
          12       0.92      0.98      0.95       690
          13       0.95      0.98      0.97       720
          14       0.92      0.83      0.87       270
          15       0.98      0.95      0.96       210
          16       0.88      0.84      0.86       150
          17       0.97    

## Analysis of results
Here we are going to make some comments about the obtained results by taking into account the analysis of notebook: **LAB1_excersize1-EDA**, in particular:

> 2. (**Section 1**) **[TBC.1]** Refers to the fact that data can be splitted into 3 *"macro-classes"* Highly Represented, Medium and Low. 
>3. (**Section 1**) Since the **[TBC.1]** *"macro-classes"* composition is coherent between training and test set I would expect a model to better perform on Highly Represented classes, a bit worst on Medium, and definitely worst on Low, since the predictive performance of the model will be driven by the data composition.

So a reminder:
- classes 5, 4, 10, 38 12, 13, 1, 2 as **Highly Represented Classes**
- classes 17, 18, 35, 11, 3, 7, 8, 9, 25 as **Medium Represented Classes**
- remaining as **low represented classes** 

Results of SVC (not fine-tuned) seems to confirm (in general) **[TBC.1]**, surprisingly **class 2** (which is the most populated) has worst results on all metrics wrt almost all other classed in **Highly Represented Classes**.

---
### Exercise 1.3: A Fine-tuning Baseline

In this exercise you should try to *improve* on the stable baseline given by the feature extraction + SVM (or whatever) classifier you produced in the previous lecture. To do this, you should *fine-tune* the ResNet-18 (or whatever model you chose) to solve the new classification task.

To do this, you could proceed by:
1. Loading the ResNet-18 (or whatever) model and replacing the final FC layer (the classifier) with a *new* classifier. This could be a single Linear layer, or could be an MLP.
2. Training the resulting model on the GTSRB dataset for a few epochs.
3. Evaluating the resulting performance.

Some things you should probably consider (especially thinking about the *next* exercise):
+ You should be monitoring not only the loss on the training set, but also a *validation* loss on an *independent* validation set. Split the training set into two datasets: one with, say, 80% of the original training samples, and another with the remaining 20%. You can use this smaller set to monitor performance and check for *overfitting*.
+ Maybe the best strategy is to not fine-tune *all* layers, but only the last few. Think about *selectively* fine-tuning layers of the network.
+ Maybe a *single* linear layer isn't the best option for the classifier. Think about using an MLP instead.

In [22]:
len(train_loader)

53

In [23]:
model = get_model('resnet18', weights = 'DEFAULT')
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
# Blocking the training of the first 2 layers of resnet
for p in model.layer1.parameters():
    p.requires_grad_(False)

for p in model.layer2.parameters():
    p.requires_grad_(False)

# Running experiments

## Important:
Experiments implementation consist of a Grid Search HyperParameters Optimization on the values<br>
in *config.yaml* file declared at **EXPERIMENTS_HYPERPARAMS**.<br>
<br>
It would not the best choice to re-run all experiments so there is a strategy for this;<br>
**EXECUTION_MODE** is a string variable that can assume following values:
> "RETRAIN", "RETRAIN_BEST", "EVAL_BEST"
- "RETRAIN": re-runs all the HPO exactly as it was done the first time
- "RETRAIN_BEST": best HPO params will be retrieved and used to retrain and test
- "EVAL_BEST": loads the best model weights and only executes inference on test set

<br>
<br>

HERE YOU CAN CHANGE THE **EXECUTION_MODE** AS YOU WISH (suggested option: "EVAL_BEST", all results could be provided anyway).

In [ ]:
# "RETRAIN": re-runs all the HPO exactly as it was done the first time
# "RETRAIN_BEST": best HPO params will be retrieved and used to retrain and test
# "EVAL_BEST": loads the best model weights and only executes inference on test set
EXECUTION_MODE = "EVAL_BEST"

In [62]:
transform = T.Compose([T.Resize(70), T.RandomCrop((64,64)), T.ToTensor(), T.Normalize(mean = [0.485, 0.456, 0.406], std=[0.229,0.224, 0.225])])
ds_train = GTSRB('_data/', split='train', transform=transform, download=True)
ds_test = GTSRB('_data/', split='test', transform=transform, download=True)
train_loader = DataLoader(ds_train, BATCH_SIZE, shuffle=True)
test_loader = DataLoader(ds_test, BATCH_SIZE, shuffle=True)

/Users/edoardocanti/opt/anaconda3/envs/DLA2026/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [63]:
model = get_model('resnet18', weights = 'DEFAULT')
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experiment 1
### Description:
Using the ```Experiment``` class defined in **utils.py** the goal is to apply a scratch implementation<br> 
of a **grid-search HPO** defined in ```config.yaml``` file (at *EXPERIMENTS_HYPERPARAMS*)<br>
by freezing the first 2 layer of resnet 18, with a **linear classification head**<br>

In [ ]:
def instanciate_model_v1():
    model = get_model('resnet18', weights = 'DEFAULT')
    for p in model.layer1.parameters():
        p.requires_grad_(False)
    for p in model.layer2.parameters():
        p.requires_grad_(False)
    model.fc = torch.nn.Linear(512, 43) # We are projecting the feature representation over the 43 dimensional space
    return model

In [ ]:
with open('../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

if EXECUTION_MODE not in ["RETRAIN", "RETRAIN_BEST", "EVAL_BEST"]:
    print("[ERROR] EXECUTION_MODE not allowed.")
elif EXECUTION_MODE == "RETRAIN":
    print("> All experiments are going to be run again, it will take a long time...")
    EXPERIMENTS_HYPERPARAMS = config["EXPERIMENTS_HYPERPARAMS"]
    num_epochs = EXPERIMENTS_HYPERPARAMS['training_epochs']
    learning_rates = EXPERIMENTS_HYPERPARAMS['learning_rates']
    batch_sizes = EXPERIMENTS_HYPERPARAMS['batch_sizes']
    early_stopping = EXPERIMENTS_HYPERPARAMS['early_stopping']
    patience = EXPERIMENTS_HYPERPARAMS['patience']
elif EXECUTION_MODE == "RETRAIN_BEST":
    EXPERIMENTS_HYPERPARAMS = config["EXPERIMENTS_HYPERPARAMS"]
    print("> Retraining again the best model")
    # Also if these are single params they must be passed as list beacause of the loop in next cell
    num_epochs = [30] 
    learning_rates = [0.001]
    batch_sizes = [128]
    early_stopping = EXPERIMENTS_HYPERPARAMS['early_stopping']
    patience = EXPERIMENTS_HYPERPARAMS['patience']
elif EXECUTION_MODE == "EVAL_BEST":
    print("> Evaluation Mode: Evaluating the best model")
    # PLEASE CHANGE THE BEST_MODEL_PATH VALUE WITH THE LOCATION OF THE WEIGHTS YOU DOWNLOADED
    BEST_MODEL_PATH = "../experiments/EP30_LR0.001_BS128/ep30_lr0.001_bs128_last_model.pth"
    batch_sizes = [128]
    

In [ ]:
from utils import Experiment

if EXECUTION_MODE == "RETRAIN" or EXECUTION_MODE == "RETRAIN_BEST":
    for epoch in num_epochs:
        for lr in learning_rates:
            for bs in batch_sizes:
                title = "ep{}_lr{}_bs{}".format(epoch, lr, bs)
                dirname = "EP{}_LR{}_BS{}".format(epoch, lr, bs)
                # maybe the name is a bit confounding:
                # original_trainset is a deep copy of the training data. Why? because Experiment class stratifies while creating the validation set
                # by only passing ds_train, what would happen (it actually happened, this comment is written posterior to the bug) is that
                # the original training data will become always smaller and smaller, because we are going to stratify on it at every experiment (the argument is actually a ref!)
                # by doing deep copy we are going to create a brand new copy of the training data.
                # Of course this kind of problem only arises if we start a multiplicity of experiments within a loop 
                # (because the training set ref is passed over and over)
                original_trainset = copy.deepcopy(ds_train)
                experiment = Experiment(training_data = original_trainset, testing_data = ds_test, title=title,
                            motivation="Freezing the first 2 layers and implementing a grid search HPO",
                            directory_name=dirname, need_valset=True, valset_proportion=0.2)
                print("STARTING EXPERIMENT {}".format(title))
                current_model = instanciate_model_v1()
                experiment.start_experiment(model = current_model, num_epochs=epoch, in_batch_size=bs, learning_rate=lr, verbose=1)
elif EXECUTION_MODE == "EVAL_BEST":
    print("Preparing evaluation mode...")
    best_model = instanciate_model_v1()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    # This part has been written after all experiments
    # I need to fake the implementation of the training data in order to use the same class Experiment
    original_trainset = copy.deepcopy(ds_train)
    experiment = Experiment(training_data=original_trainset, testing_data=ds_test, 
                            title="EVALUATION MODE", 
                            motivation="Final evaluation on test set",
                            directory_name="EVAL", 
                            need_valset=False)
    experiment.connect_model(best_model)
    eval_bs = batch_sizes[0]
    experiment.allocate_dataloaders(eval_bs)
    print("Running evaluation on test set...")
    acc, report, loss = experiment.evaluate(experiment.test_loader, verbose=False)
    print("TEST METRICS IN EVALUATION")
    print("Accuracy: {}".format(acc))
    print("Loss: {}".format(loss))
    print("Classification Report:")
    print(report)
    

STARTING EXPERIMENT ep10_lr0.01_bs128
[WARNING] No optimizer provided, setting Adam default
Training started on 10 total epochs with learning_rate 0.01
------------------------------
> Training epoch 1/10
> Epoch: 1/10 Loss: 1.3284261155271244
> Epoch: 1/10 Validation Loss: 0.6156171311934789
------------------------------
> Training epoch 2/10
> Epoch: 2/10 Loss: 0.23199326431233727
> Epoch: 2/10 Validation Loss: 0.16022125862184025
------------------------------
> Training epoch 3/10
> Epoch: 3/10 Loss: 0.11051209947975453
> Epoch: 3/10 Validation Loss: 0.7238080125479471
------------------------------
> Training epoch 4/10
> Epoch: 4/10 Loss: 0.07251235691573985
> Epoch: 4/10 Validation Loss: 0.13403219747401418
------------------------------
> Training epoch 5/10
> Epoch: 5/10 Loss: 0.057476806168550415
> Epoch: 5/10 Validation Loss: 0.09133157950072061
------------------------------
> Training epoch 6/10
> Epoch: 6/10 Loss: 0.0502323985175079
> Epoch: 6/10 Validation Loss: 0.11591

## Experiment 1 - Analysis of results
**Best result:** EP30_LR0.001_BS128<br>
The best result has been obtained using 30 training epochs, with an Adam Optimizer equipped with a learning rate of 0.001 and a batch size of 128.<br>
The model reached a test accuracy of 0.947.<br>
The **Classification report** shows some coherent aspects wrt the *Exploratory Data Analysis* reported in **LAB1_ecercise1-EDA.ipynb**:<br>
- The model performs very well on the so called: **Highly Represented Classes** (2,1,13,12,38,10,4,5)
- The **weighted avg** is slightly better than **macro**, this means that the model performs well on the most represented classes, while low represented classes tends to lower the macro average (classification report of classes: 19, 27, 28, 29, 30...)
- The **gap** between **weighted avg* *macro avg** also confirms what has been stated during flipped lecture:
> "we can treat the problem as a not **too** imbalanced one"
By taking a look at the **loss_chart.png** we can say that the general trend is incresing in performances, also if we can identify a non-negligeble presence of spikes.<br>
By the way, since the spikes seems to be counterbalanced by local minima, we can conclude that the model is not going to overfit (at least at epoch 30).
<br>
<br>

The conclusion is that a Fine-Tuned ResNet-18 with *Linear layer* head, is enough to achieve *good* performances. this suggest that the backbone **ResNet18** has a representation power, strong enough, to make a Linear model a good choice.<br>
Also, the fact that the ResNet-18 was fine-tuned, by freezing the first two layers, I think shows the following fact; as known the first layers of a CNN can learn simple filters, while successive layers tend to learn more complext patterns, probably the original data on which the first two layers of ResNet18 were trained, has been enough to learn patterns that, *arriving at the Linear model head* were already learnt.


## Experiment 2 (kind of bonus)
### Description:
Using the ```Experiment``` class defined in **utils.py** the goal is to apply a scratch implementation<br> 
of a **grid-search HPO** defined in ```config.yaml``` file (at *MLP_EXPERIMENT_HYPERPARAMS*)<br>
by freezing the first 2 layer of resnet 18, with a **MultiLayer Perceptron**<br>
with the possibility of using different activations functions.

In [10]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/edoardocanti/.netrc.
wandb: Currently logged in as: edoardo-canti (edoardo-canti-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from utils import MLP
with open("../config.yaml", 'r') as file:
    config = yaml.safe_load(file)


MLP_HP = config["MLP"]
INPUT_DIM = MLP_HP["input_layer_nodes"]
HIDDEN_DIM = MLP_HP["hidden_layer_nodes"]
OUTPUT_DIM = MLP_HP["output_layer_nodes"]
WEIGHT_INIT = MLP_HP["weight_init"]
ACT_FUNC = MLP_HP["activation_function"]

print("MLP config:\n  " \
"Input dimension: {}\n  " \
"Hidden dimension: {}\n  " \
"Output dimension: {}\n  " \
"Weights init method: {}\n  " \
"Activation function: {}".format(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, WEIGHT_INIT, ACT_FUNC))


MLP_EXP_HYPERPARAMS = config["MLP_EXPERIMENT_HYPERPARAMS"]
#experiments hyperparams
num_epoch = MLP_EXP_HYPERPARAMS['training_epochs']
lr = MLP_EXP_HYPERPARAMS['learning_rate']
bs = MLP_EXP_HYPERPARAMS['batch_size']
es = MLP_EXP_HYPERPARAMS['early_stopping']
patience = MLP_EXP_HYPERPARAMS['patience']

print("="*60)
print("Experiment is going to be performed on following config --> epochs: {}, lr: {}, batch_size: {}, early_stopping:{}, patience:{}".format(num_epoch, lr, bs, es,patience))

MLP config:
  Input dimension: 512
  Hidden dimension: 256
  Output dimension: 43
  Weights init method: he_uniform
  Activation function: relu
Experiment is going to be performed on following config --> epochs: 15, lr: 0.001, batch_size: 128, early_stopping:True, patience:5


In [23]:
def instanciate_model_v2(input_dim, hidden_dim, output_dim, w_init, act_func):
    model = get_model('resnet18', weights='DEFAULT')
    for p in model.layer1.parameters(): p.requires_grad_(False)
    for p in model.layer2.parameters(): p.requires_grad_(False)
    model.fc = MLP(input_dim, hidden_dim, output_dim, 
                   weight_init=w_init, 
                   activation_function=act_func)
    return model

### He uniform weights initializatio and ReLU

In [24]:
from utils import Experiment
import copy

mlp_base_dir = os.path.join("experiments", "MLP_WANDB")
title = "ep{}_lr{}_bs{}".format(num_epoch, lr, bs)
dirname = "EP{}_LR{}_BS{}".format(num_epoch, lr, bs)
experiment =  Experiment(training_data=copy.deepcopy(ds_train), testing_data=ds_test, 
                                     title=title, motivation="MLP with best HP of FT-ResNet18 fine tuned and ReLU", 
                                     directory_name=dirname, base_dir=mlp_base_dir, 
                                     need_valset=True, valset_proportion=0.2,
                                     use_wandb=True, wandb_project="ResNet_MLP_Project ReLU" )

resnet_mlp = instanciate_model_v2(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, WEIGHT_INIT, ACT_FUNC)
            
experiment.start_experiment(model = resnet_mlp, num_epochs=num_epoch, in_batch_size=bs, learning_rate=lr, verbose=1)

[WARNING] No optimizer provided, setting Adam default
Training started on 15 total epochs with learning_rate 0.001
------------------------------
> Training epoch 1/15
> Epoch: 1/15 Loss: 0.5604469496868327
> Epoch: 1/15 Validation Loss: 0.20362140761599654
------------------------------
> Training epoch 2/15
> Epoch: 2/15 Loss: 0.10504858664231385
> Epoch: 2/15 Validation Loss: 0.08821190068764347
------------------------------
> Training epoch 3/15
> Epoch: 3/15 Loss: 0.07337966571076812
> Epoch: 3/15 Validation Loss: 0.07503092139294106
------------------------------
> Training epoch 4/15
> Epoch: 4/15 Loss: 0.059211717597850246
> Epoch: 4/15 Validation Loss: 0.0391350334310638
------------------------------
> Training epoch 5/15
> Epoch: 5/15 Loss: 0.03672730177362076
> Epoch: 5/15 Validation Loss: 0.03940406999373365
------------------------------
> Training epoch 6/15
> Epoch: 6/15 Loss: 0.03545947371798004
> Epoch: 6/15 Validation Loss: 0.08048442365335566
----------------------

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_accuracy,▁
test_loss,▁
train_loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_loss,▅▂▂▁▁▂▂█▁▁▃▁▂▃▁
epoch,15
test_accuracy,0.94181
test_loss,0.24913
train_loss,0.01862
val_loss,0.02311


>>> Experiment Ended
